# Lab 3: Actor-Critic Methods

## TDDE78 — Deep Reinforcement Learning
### Linköping University, Spring 2026

---

In this lab you will implement two fundamental **actor-critic** algorithms:
- **A2C (Advantage Actor-Critic with GAE)** — an on-policy method combining rollout-based data collection with Generalized Advantage Estimation
- **SAC (Soft Actor-Critic)** — a state-of-the-art off-policy algorithm for continuous control with entropy regularization and twin Q-networks

The lab is divided into:
- **Part A** — Implementation (fill in the TODO sections)
- **Part B** — Experiments (run systematic evaluations and analyze results)

**Instructions:** Complete all cells marked with `# TODO`. Do not modify the provided helper code unless explicitly asked.

## Setup

Run the cell below to import all necessary libraries and verify your environment.

In [ ]:
import os
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import gymnasium as gym
import matplotlib.pyplot as plt
from IPython.display import Video, display
import warnings
warnings.filterwarnings('ignore')

# Import lab utilities
from networks import ContinuousActorCritic, SACActor, SACCritic
from utils import compute_gae, RolloutBuffer, ReplayBuffer, plot_a2c_results, plot_sac_results, plot_comparison, record_agent_video, smooth

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Resolve experiments directory relative to this notebook's location
_here = globals().get('__vsc_ipynb_file__', os.path.abspath(''))
_nb_dir = os.path.dirname(_here) if os.path.isfile(_here) else _here
EXPERIMENTS_DIR = os.path.normpath(os.path.join(_nb_dir, '..', 'experiments'))
print(f"Experiments directory: {EXPERIMENTS_DIR}")

# For reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

set_seed(42)
print("Setup complete!")

---

# Part A — Implementation

---

## A.1 — Explore the Environment

We'll work with **LunarLanderContinuous-v3** — a continuous control task where the agent must land a spacecraft between two flags.

- **State:** 8-dimensional vector (position x/y, velocity x/y, angle, angular velocity, leg contacts)
- **Actions:** 2 continuous values — main engine thrust and side engine thrust, both in \([-1, 1]\)
- **Reward:** +100 for landing, -100 for crash, small rewards for each leg contact, penalty for using fuel
- **Solved:** Average reward ≥ 200 over 100 consecutive episodes

In [ ]:
# Explore the LunarLanderContinuous environment
env = gym.make('LunarLanderContinuous-v3')

print(f"Observation space: {env.observation_space}")
print(f"  Shape: {env.observation_space.shape}")
print(f"  Low:   {env.observation_space.low}")
print(f"  High:  {env.observation_space.high}")
print()
print(f"Action space: {env.action_space}")
print(f"  Shape: {env.action_space.shape}")
print(f"  Low:   {env.action_space.low}")
print(f"  High:  {env.action_space.high}")

# Run a random episode
obs, info = env.reset(seed=42)
total_reward = 0
for step in range(1000):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    if terminated or truncated:
        break

print(f"\nRandom agent — Episode length: {step+1}, Total reward: {total_reward:.1f}")
env.close()

## A.2 — Utility Functions (GAE & Buffers)

Open `utils.py` and implement:
1. **`compute_gae()`** — Generalized Advantage Estimation computed backwards in time
2. **`RolloutBuffer.store()`** and **`RolloutBuffer.compute_returns_and_advantages()`** — on-policy buffer for A2C
3. **`ReplayBuffer.push()`** and **`ReplayBuffer.sample()`** — off-policy buffer for SAC

**Why GAE?** Pure Monte Carlo returns have zero bias but very high variance. One-step TD is low variance but high bias. GAE with \(\lambda \in (0,1)\) interpolates between them:

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t), \qquad A_t = \sum_{l=0}^{\infty}(\gamma\lambda)^l \delta_{t+l}$$

In [ ]:
# Test compute_gae
from utils import compute_gae

# Simple 3-step example: rewards=[1,1,1], values=[0,0,0], done=[F,F,T], last_value=0
rewards    = [1.0, 1.0, 1.0]
values     = [0.0, 0.0, 0.0]
dones      = [False, False, True]
last_value = 0.0
gamma, lam = 0.99, 0.95

advs, rets = compute_gae(rewards, values, dones, last_value, gamma, lam)

assert advs.shape == (3,), f"Expected shape (3,), got {advs.shape}"
assert rets.shape  == (3,), f"Expected shape (3,), got {rets.shape}"
assert advs.dtype  == np.float32

# With lambda=1 and zero values, advantages equal MC returns
advs_mc, _ = compute_gae(rewards, values, dones, last_value, gamma, gae_lambda=1.0)
expected_G  = np.array([1 + 0.99 + 0.99**2, 1 + 0.99, 1.0], dtype=np.float32)
np.testing.assert_allclose(advs_mc, expected_G, rtol=1e-5,
                            err_msg="Lambda=1 should give MC returns")

print("compute_gae tests passed!")
print(f"  advantages (λ=0.95): {advs}")
print(f"  returns    (λ=0.95): {rets}")
print(f"  advantages (λ=1.0 ): {advs_mc}  ← should equal MC returns {expected_G}")

In [ ]:
# Test RolloutBuffer
from utils import RolloutBuffer

buf = RolloutBuffer(n_steps=10, state_dim=8, action_dim=2)
for i in range(10):
    buf.store(
        state    = np.zeros(8, dtype=np.float32),
        action   = np.array([0.1, -0.1], dtype=np.float32),
        reward   = 1.0,
        done     = (i == 9),
        log_prob = -1.0,
        value    = 0.5,
    )

assert len(buf) == 10, f"Expected 10, got {len(buf)}"

buf.compute_returns_and_advantages(last_value=0.0, gamma=0.99, gae_lambda=0.95)
assert buf.advantages is not None and buf.returns is not None, "advantages/returns not set"

batches = list(buf.get_batches(batch_size=5))
assert len(batches) == 2, f"Expected 2 batches, got {len(batches)}"
states, actions, old_log_probs, advantages, returns = batches[0]
assert states.shape    == (5, 8), f"states: {states.shape}"
assert actions.shape   == (5, 2), f"actions: {actions.shape}"
assert advantages.shape == (5,),  f"advantages: {advantages.shape}"

print("RolloutBuffer tests passed!")

# Test ReplayBuffer
from utils import ReplayBuffer

rb = ReplayBuffer(capacity=1000, state_dim=8, action_dim=2, seed=42)
for i in range(100):
    rb.push(
        state      = np.zeros(8, dtype=np.float32),
        action     = np.array([0.1, -0.1], dtype=np.float32),
        reward     = 1.0,
        next_state = np.ones(8, dtype=np.float32),
        done       = False,
    )
assert len(rb) == 100

states, actions, rewards, next_states, dones = rb.sample(batch_size=32)
assert states.shape      == (32, 8), f"states: {states.shape}"
assert actions.shape     == (32, 2), f"actions: {actions.shape}"
assert rewards.shape     == (32, 1), f"rewards should be (32,1), got {rewards.shape}"
assert next_states.shape == (32, 8), f"next_states: {next_states.shape}"
assert dones.shape       == (32, 1), f"dones should be (32,1), got {dones.shape}"

print("ReplayBuffer tests passed!")

## A.3 — Network Architectures

Open `networks.py` and implement the three network classes:

**`ContinuousActorCritic`** (for A2C):
```
state_dim → 256 → Tanh → 256 → Tanh
                                    ├── → action_dim   [actor: mean μ(s)]
                                    └── → 1            [critic: V(s)]
log_std: nn.Parameter(zeros)  [learnable, shared across states]
```

**`SACActor`** (for SAC — reparameterized Gaussian with tanh squashing):
```
state_dim → 256 → ReLU → 256 → ReLU
                                     ├── → action_dim  [mean μ(s)]
                                     └── → action_dim  [log std log σ(s), clamped]
```

**`SACCritic`** (for SAC — twin Q-networks):
```
[state | action] → 256 → ReLU → 256 → ReLU → 1   (×2 independent networks)
```

In [ ]:
# Test ContinuousActorCritic
from networks import ContinuousActorCritic

net = ContinuousActorCritic(state_dim=8, action_dim=2).to(device)
test_states = torch.randn(4, 8).to(device)

mean, value = net(test_states)
assert mean.shape  == (4, 2), f"mean shape: {mean.shape}"
assert value.shape == (4, 1), f"value shape: {value.shape}"

action, log_prob, entropy, value2 = net.get_action(test_states)
assert action.shape   == (4, 2), f"action shape: {action.shape}"
assert log_prob.shape == (4,),   f"log_prob shape: {log_prob.shape}"
assert entropy.shape  == (4,),   f"entropy shape: {entropy.shape}"

n_params = sum(p.numel() for p in net.parameters())
print(f"ContinuousActorCritic — parameters: {n_params:,}")
print(f"  mean: {mean.shape}, value: {value.shape}")
print("ContinuousActorCritic tests passed!")

# Test SACActor
from networks import SACActor

actor = SACActor(state_dim=8, action_dim=2).to(device)
action, log_prob = actor.get_action(test_states)
assert action.shape   == (4, 2), f"action shape: {action.shape}"
assert log_prob.shape == (4,),   f"log_prob shape: {log_prob.shape}"
assert action.abs().max().item() < 1.0, "SAC actions must be in (-1,1) after tanh"

n_params = sum(p.numel() for p in actor.parameters())
print(f"\nSACActor — parameters: {n_params:,}")
print(f"  action range: [{action.min().item():.3f}, {action.max().item():.3f}]  ← must be in (-1,1)")
print("SACActor tests passed!")

# Test SACCritic
from networks import SACCritic

critic = SACCritic(state_dim=8, action_dim=2).to(device)
test_actions = torch.randn(4, 2).to(device)
q1, q2 = critic(test_states, test_actions)
assert q1.shape == (4, 1), f"q1 shape: {q1.shape}"
assert q2.shape == (4, 1), f"q2 shape: {q2.shape}"

n_params = sum(p.numel() for p in critic.parameters())
print(f"\nSACCritic (twin) — parameters: {n_params:,}")
print(f"  q1: {q1.shape}, q2: {q2.shape}")
print("SACCritic tests passed!")

## A.4 — A2C Agent

Implement the complete A2C agent. Key steps:
1. **`collect_rollout()`** — run the policy for `n_steps` steps and store transitions
2. **`update()`** — compute GAE advantages then update actor and critic with a single gradient step

**A2C Loss:**
$$\mathcal{L} = \underbrace{-\mathbb{E}[\log\pi_\theta(a|s) \cdot \hat{A}]}_{\text{policy loss}} + \underbrace{c_V \cdot \mathbb{E}[(V(s) - R)^2]}_{\text{value loss}} - \underbrace{c_H \cdot \mathcal{H}[\pi]}_{\text{entropy bonus}}$$

In [ ]:
class A2CAgent:
    """
    Advantage Actor-Critic (A2C) agent for continuous control.

    Collects n_steps transitions, computes GAE advantages, then performs
    one gradient update on the actor and critic simultaneously.

    Args:
        state_dim     (int):   Dimension of state space.
        action_dim    (int):   Dimension of continuous action space.
        lr            (float): Learning rate.
        gamma         (float): Discount factor.
        gae_lambda    (float): GAE lambda parameter.
        n_steps       (int):   Steps per rollout (one update per rollout).
        batch_size    (int):   Mini-batch size within each rollout.
        vf_coef       (float): Coefficient for value loss.
        ent_coef      (float): Coefficient for entropy bonus.
        max_grad_norm (float): Gradient clipping norm.
        seed          (int):   Random seed.
    """

    def __init__(
        self,
        state_dim: int,
        action_dim: int,
        lr: float = 3e-4,
        gamma: float = 0.99,
        gae_lambda: float = 0.95,
        n_steps: int = 2048,
        batch_size: int = 64,
        vf_coef: float = 0.5,
        ent_coef: float = 0.0,
        max_grad_norm: float = 0.5,
        seed: int = 42,
    ):
        self.gamma         = gamma
        self.gae_lambda    = gae_lambda
        self.n_steps       = n_steps
        self.batch_size    = batch_size
        self.vf_coef       = vf_coef
        self.ent_coef      = ent_coef
        self.max_grad_norm = max_grad_norm

        # =====================================================================
        # TODO: Initialize the following:
        #
        # 1. self.network   — ContinuousActorCritic(state_dim, action_dim) moved to device
        # 2. self.optimizer — Adam optimizer for self.network with lr
        # 3. self.buffer    — RolloutBuffer(n_steps, state_dim, action_dim)
        # =====================================================================
        raise NotImplementedError("Initialize A2CAgent components")

    @torch.no_grad()
    def collect_rollout(self, env, obs):
        """
        Collect n_steps transitions from the environment using the current policy.

        Args:
            env: Gymnasium environment (already reset; obs is the current state).
            obs: Current observation as numpy array of shape (state_dim,).

        Returns:
            obs:             Next observation after the rollout ends.
            last_value:      V(s_T) — bootstrapped value for the final state.
            episode_rewards: List of total rewards for any episodes completed.
        """
        self.buffer.clear()
        episode_rewards = []
        current_ep_reward = 0.0

        # =====================================================================
        # TODO: Implement the rollout collection loop.
        #
        # For each step in range(self.n_steps):
        #   1. Convert obs to tensor:
        #        state_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
        #   2. Sample action from the policy:
        #        action, log_prob, entropy, value = self.network.get_action(state_t)
        #        action_np   = action.cpu().numpy().flatten()
        #        log_prob_val = log_prob.item()
        #        value_val    = value.item()
        #   3. Step the environment:
        #        next_obs, reward, terminated, truncated, _ = env.step(action_np)
        #        done = terminated or truncated
        #   4. Store transition:
        #        self.buffer.store(obs, action_np, reward, done, log_prob_val, value_val)
        #   5. Track episode reward; on done append to episode_rewards and reset
        #   6. obs = next_obs
        #
        # After the loop, bootstrap last_value:
        #   state_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
        #   last_value = self.network(state_t)[1].item()  (value head)
        #   If the last step was a terminal (done=True), last_value = 0.0
        #
        # Return obs, last_value, episode_rewards
        # =====================================================================
        raise NotImplementedError("Implement A2CAgent.collect_rollout()")

    def update(self):
        """
        Perform one A2C gradient update using the data stored in self.buffer.

        compute_returns_and_advantages() must be called on self.buffer BEFORE
        calling this method (done in the training loop).

        Returns:
            dict with keys 'policy_loss', 'value_loss', 'entropy'
        """
        policy_losses, value_losses, entropies = [], [], []

        # =====================================================================
        # TODO: Implement the A2C update.
        #
        # For each mini-batch from self.buffer.get_batches(self.batch_size):
        #   1. Unpack: states, actions, old_log_probs, advantages, returns
        #   2. Move all tensors to device
        #   3. Normalize advantages (in-place or new tensor):
        #        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        #   4. Re-evaluate the stored actions under the CURRENT policy:
        #        _, log_probs, entropy, values = self.network.get_action(states, actions)
        #   5. Compute losses:
        #        policy_loss = -(log_probs * advantages).mean()
        #        value_loss  = F.mse_loss(values.squeeze(-1), returns)
        #        entropy_loss = -entropy.mean()
        #        total_loss  = policy_loss + self.vf_coef * value_loss + self.ent_coef * entropy_loss
        #   6. Backprop + clip + step:
        #        self.optimizer.zero_grad()
        #        total_loss.backward()
        #        nn.utils.clip_grad_norm_(self.network.parameters(), self.max_grad_norm)
        #        self.optimizer.step()
        #   7. Record loss values
        # =====================================================================
        raise NotImplementedError("Implement A2CAgent.update()")

print("A2CAgent class defined!")

In [ ]:
def train_a2c(
    env_name: str = "LunarLanderContinuous-v3",
    total_timesteps: int = 1_000_000,
    seed: int = 42,
    solve_threshold: float = None,
    log_interval: int = 10,
    **agent_kwargs,
):
    """
    Train an A2C agent and return training metrics.

    Args:
        env_name:         Gymnasium environment name.
        total_timesteps:  Total environment steps to train for.
        seed:             Random seed.
        solve_threshold:  Stop early when avg reward (last 10 episodes) >= this.
        log_interval:     Print progress every this many rollout updates.
        **agent_kwargs:   Passed to A2CAgent (e.g. gae_lambda, ent_coef).

    Returns:
        dict with keys: 'episode_rewards', 'episode_timesteps',
                        'policy_losses', 'value_losses', 'agent'
    """
    set_seed(seed)

    env = gym.make(env_name)
    state_dim  = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]

    agent = A2CAgent(state_dim=state_dim, action_dim=action_dim, seed=seed, **agent_kwargs)

    episode_rewards   = []
    episode_timesteps = []
    policy_losses     = []
    value_losses      = []
    global_step       = 0
    update_num        = 0

    obs, _ = env.reset(seed=seed)

    # =====================================================================
    # TODO: Implement the A2C training loop.
    #
    # while global_step < total_timesteps:
    #   1. Collect rollout:
    #        obs, last_value, ep_rewards = agent.collect_rollout(env, obs)
    #        global_step += agent.n_steps
    #   2. Compute advantages:
    #        agent.buffer.compute_returns_and_advantages(last_value, agent.gamma, agent.gae_lambda)
    #   3. Update the network:
    #        metrics = agent.update()
    #        update_num += 1
    #   4. Record completed episode rewards from ep_rewards
    #        for r in ep_rewards:
    #            episode_rewards.append(r)
    #            episode_timesteps.append(global_step)
    #   5. Record losses:
    #        policy_losses.append(metrics['policy_loss'])
    #        value_losses.append(metrics['value_loss'])
    #   6. Log every log_interval updates:
    #        avg_reward = np.mean(episode_rewards[-10:]) if episode_rewards else 0.0
    #        print(f"Update {update_num} | Steps {global_step:,} | ...")
    #   7. Check solve_threshold early stopping
    # =====================================================================
    raise NotImplementedError("Implement A2C training loop")

    env.close()
    return {
        'episode_rewards':   episode_rewards,
        'episode_timesteps': episode_timesteps,
        'policy_losses':     policy_losses,
        'value_losses':      value_losses,
        'agent':             agent,
    }

print("train_a2c defined!")

In [ ]:
class SACAgent:
    """
    Soft Actor-Critic (SAC) agent for continuous control.

    Off-policy maximum entropy actor-critic with:
    - Twin Q-networks (clipped double-Q to reduce overestimation)
    - Automatic entropy temperature tuning
    - Soft target network updates (polyak averaging)

    Args:
        state_dim       (int):   Dimension of state space.
        action_dim      (int):   Dimension of continuous action space.
        lr              (float): Learning rate for actor, critics, and alpha.
        gamma           (float): Discount factor.
        tau             (float): Soft target update coefficient.
        alpha           (float): Initial entropy temperature (used if not auto-tuning).
        autotune_alpha  (bool):  Automatically tune alpha to match target entropy.
        target_entropy  (float): Target entropy for auto-tuning. Default: -action_dim.
        buffer_size     (int):   Replay buffer capacity.
        batch_size      (int):   Mini-batch size per gradient step.
        learning_starts (int):   Environment steps before learning begins.
        train_frequency (int):   Gradient steps per environment step.
        seed            (int):   Random seed.
    """

    def __init__(
        self,
        state_dim: int,
        action_dim: int,
        lr: float = 3e-4,
        gamma: float = 0.99,
        tau: float = 0.005,
        alpha: float = 0.2,
        autotune_alpha: bool = True,
        target_entropy: float = None,
        buffer_size: int = 1_000_000,
        batch_size: int = 256,
        learning_starts: int = 5000,
        train_frequency: int = 1,
        seed: int = 42,
    ):
        self.gamma           = gamma
        self.tau             = tau
        self.batch_size      = batch_size
        self.learning_starts = learning_starts
        self.train_frequency = train_frequency
        self.autotune_alpha  = autotune_alpha

        # =====================================================================
        # TODO: Initialize the following:
        #
        # 1. self.actor  — SACActor(state_dim, action_dim) on device
        # 2. self.critic — SACCritic(state_dim, action_dim) on device
        # 3. self.critic_target — SACCritic(state_dim, action_dim) on device,
        #      copy weights from self.critic, set to eval mode
        # 4. self.actor_optimizer  — optim.Adam(self.actor.parameters(), lr=lr)
        # 5. self.critic_optimizer — optim.Adam(self.critic.parameters(), lr=lr)
        # 6. self.replay_buffer — ReplayBuffer(buffer_size, state_dim, action_dim, seed)
        #
        # 7. Entropy temperature:
        #    if autotune_alpha:
        #        self.target_entropy  = float(target_entropy) if target_entropy else -action_dim
        #        self.log_alpha       = torch.zeros(1, requires_grad=True, device=device)
        #        self.alpha           = self.log_alpha.exp().item()
        #        self.alpha_optimizer = optim.Adam([self.log_alpha], lr=lr)
        #    else:
        #        self.alpha = alpha
        # =====================================================================
        raise NotImplementedError("Initialize SACAgent components")

    def select_action(self, state, deterministic=False):
        """
        Select an action given a state observation.

        Args:
            state:         numpy array of shape (state_dim,)
            deterministic: If True, use tanh(mean) — no noise, for evaluation.

        Returns:
            numpy array of shape (action_dim,)
        """
        # =====================================================================
        # TODO: Implement action selection.
        #
        # 1. state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        # 2. with torch.no_grad():
        #      if deterministic:
        #          mean, _ = self.actor(state_t)
        #          action  = torch.tanh(mean)
        #      else:
        #          action, _ = self.actor.get_action(state_t)
        # 3. return action.cpu().numpy().flatten()
        # =====================================================================
        raise NotImplementedError("Implement SACAgent.select_action()")

    def update(self):
        """
        Perform one full SAC gradient update:
          1. Critic update (both Q-networks)
          2. Actor update
          3. Temperature update (if autotune_alpha)
          4. Soft target network update

        Returns:
            dict with keys 'critic_loss', 'actor_loss', 'alpha_loss', 'alpha'
        """
        # =====================================================================
        # TODO: Implement the full SAC update.
        #
        # Step 1 — Sample a batch from replay buffer:
        #   states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)
        #   (Move all to device; they are already torch tensors from ReplayBuffer.sample())
        #
        # Step 2 — Critic update:
        #   with torch.no_grad():
        #     next_actions, next_log_pi = self.actor.get_action(next_states)
        #     q1_next, q2_next = self.critic_target(next_states, next_actions)
        #     min_q_next = torch.min(q1_next, q2_next) - self.alpha * next_log_pi.unsqueeze(-1)
        #     y = rewards + self.gamma * (1 - dones) * min_q_next
        #
        #   q1, q2 = self.critic(states, actions)
        #   critic_loss = F.mse_loss(q1, y) + F.mse_loss(q2, y)
        #   self.critic_optimizer.zero_grad()
        #   critic_loss.backward()
        #   self.critic_optimizer.step()
        #
        # Step 3 — Actor update:
        #   new_actions, log_pi = self.actor.get_action(states)
        #   q1_pi, q2_pi = self.critic(states, new_actions)
        #   min_q_pi = torch.min(q1_pi, q2_pi)
        #   actor_loss = (self.alpha * log_pi - min_q_pi.squeeze(-1)).mean()
        #   self.actor_optimizer.zero_grad()
        #   actor_loss.backward()
        #   self.actor_optimizer.step()
        #
        # Step 4 — Temperature update (if autotune_alpha):
        #   with torch.no_grad():
        #     _, log_pi = self.actor.get_action(states)
        #   alpha_loss = (-self.log_alpha.exp() * (log_pi + self.target_entropy)).mean()
        #   self.alpha_optimizer.zero_grad()
        #   alpha_loss.backward()
        #   self.alpha_optimizer.step()
        #   self.alpha = self.log_alpha.exp().item()
        #
        # Step 5 — Soft target update (polyak averaging):
        #   for param, target_param in zip(self.critic.parameters(),
        #                                  self.critic_target.parameters()):
        #       target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)
        #
        # Return metrics dict
        # =====================================================================
        raise NotImplementedError("Implement SACAgent.update()")

print("SACAgent class defined!")

In [ ]:
def train_sac(
    env_name: str = "LunarLanderContinuous-v3",
    total_timesteps: int = 300_000,
    seed: int = 42,
    solve_threshold: float = None,
    eval_frequency: int = 10_000,
    eval_episodes: int = 10,
    **agent_kwargs,
):
    """
    Train a SAC agent and return training metrics.

    Args:
        env_name:         Gymnasium environment name.
        total_timesteps:  Total environment steps.
        seed:             Random seed.
        solve_threshold:  Stop early when eval reward >= this value.
        eval_frequency:   Evaluate the greedy policy every this many steps.
        eval_episodes:    Number of greedy episodes per evaluation.
        **agent_kwargs:   Passed to SACAgent.

    Returns:
        dict with keys: 'episode_rewards', 'eval_rewards', 'eval_timesteps',
                        'critic_losses', 'actor_losses', 'alphas', 'agent'
    """
    set_seed(seed)

    env      = gym.make(env_name)
    eval_env = gym.make(env_name)
    state_dim  = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]

    agent = SACAgent(state_dim=state_dim, action_dim=action_dim, seed=seed, **agent_kwargs)

    episode_rewards = []
    eval_rewards    = []
    eval_timesteps  = []
    critic_losses   = []
    actor_losses    = []
    alphas          = []

    # =====================================================================
    # TODO: Implement the SAC training loop.
    #
    # obs, _ = env.reset(seed=seed)
    # episode_reward = 0
    #
    # for step in range(total_timesteps):
    #
    #   1. Select action:
    #        if step < agent.learning_starts:
    #            action = env.action_space.sample()
    #        else:
    #            action = agent.select_action(obs)
    #
    #   2. Step the environment:
    #        next_obs, reward, terminated, truncated, _ = env.step(action)
    #        done = terminated or truncated
    #
    #   3. Store transition:
    #        agent.replay_buffer.push(obs, action, reward, next_obs, float(done))
    #        obs = next_obs
    #        episode_reward += reward
    #
    #   4. On done: record episode reward, reset env
    #        if done:
    #            episode_rewards.append(episode_reward)
    #            episode_reward = 0
    #            obs, _ = env.reset()
    #
    #   5. Gradient updates (after learning_starts, every train_frequency steps):
    #        if step >= agent.learning_starts and step % agent.train_frequency == 0:
    #            if len(agent.replay_buffer) >= agent.batch_size:
    #                metrics = agent.update()
    #                critic_losses.append(metrics['critic_loss'])
    #                actor_losses.append(metrics['actor_loss'])
    #                alphas.append(metrics['alpha'])
    #
    #   6. Evaluate greedy policy every eval_frequency steps:
    #        if (step + 1) % eval_frequency == 0:
    #            eval_reward = evaluate_greedy(agent, eval_env, eval_episodes, seed)
    #            eval_rewards.append(eval_reward)
    #            eval_timesteps.append(step + 1)
    #            print(f"Step {step+1:,} | Eval reward: {eval_reward:.1f} | Alpha: {agent.alpha:.4f}")
    #            if solve_threshold and eval_reward >= solve_threshold:
    #                print(f"Solved at step {step+1}!")
    #                break
    # =====================================================================
    raise NotImplementedError("Implement SAC training loop")

    env.close()
    eval_env.close()
    return {
        'episode_rewards': episode_rewards,
        'eval_rewards':    eval_rewards,
        'eval_timesteps':  eval_timesteps,
        'critic_losses':   critic_losses,
        'actor_losses':    actor_losses,
        'alphas':          alphas,
        'agent':           agent,
    }


def evaluate_greedy(agent, env, num_episodes=10, seed=0):
    """Evaluate a SAC agent deterministically (no exploration noise)."""
    rewards = []
    for ep in range(num_episodes):
        obs, _ = env.reset(seed=seed + ep)
        ep_reward = 0
        for _ in range(10000):
            action = agent.select_action(obs, deterministic=True)
            obs, reward, terminated, truncated, _ = env.step(action)
            ep_reward += reward
            if terminated or truncated:
                break
        rewards.append(ep_reward)
    return np.mean(rewards)


print("train_sac and evaluate_greedy defined!")

In [ ]:
# Plotting helpers (plot_a2c_results, plot_sac_results, plot_comparison)
# and record_agent_video are imported from utils.py above.
# Pass experiments_dir=EXPERIMENTS_DIR to any plot/record call to auto-save to disk.
print("Utilities ready — imported from utils.py.")

In [ ]:
# A.9 — Train A2C on LunarLanderContinuous-v3
# Expected: avg reward >= 200 within ~1M environment steps
results_a2c = train_a2c(
    env_name        = "LunarLanderContinuous-v3",
    total_timesteps = 1_000_000,
    n_steps         = 2048,
    batch_size      = 64,
    lr              = 3e-4,
    gae_lambda      = 0.95,
    solve_threshold = 200.0,
    seed            = 42,
    log_interval    = 10,
)

plot_a2c_results(results_a2c,
                 title="A2C — LunarLanderContinuous-v3",
                 experiments_dir=EXPERIMENTS_DIR)

last_ep = results_a2c['episode_rewards'][-50:]
print(f"\nAverage reward (last 50 episodes): {np.mean(last_ep):.1f}")
if np.mean(last_ep) >= 200:
    print("LunarLander SOLVED with A2C! (avg >= 200)")
else:
    print("Not yet solved — check your implementation.")

In [ ]:
# A.9 — Visualize the trained A2C agent
# Record 3 episodes using the deterministic policy (tanh of the Gaussian mean).
video_path = record_agent_video(
    results_a2c['agent'],
    env_name        = "LunarLanderContinuous-v3",
    agent_type      = "a2c",
    num_episodes    = 3,
    seed            = 0,
    name_prefix     = "a2c_lunar",
    experiments_dir = EXPERIMENTS_DIR,
)
if video_path:
    display(Video(video_path, embed=True, width=500))

In [ ]:
# A.10 — Train SAC on LunarLanderContinuous-v3
# Expected: SAC is far more sample-efficient — should solve in ~200k steps
results_sac = train_sac(
    env_name        = "LunarLanderContinuous-v3",
    total_timesteps = 300_000,
    learning_starts = 5000,
    batch_size      = 256,
    lr              = 3e-4,
    tau             = 0.005,
    autotune_alpha  = True,
    solve_threshold = 200.0,
    seed            = 42,
    log_interval    = 10_000,
)

plot_sac_results(results_sac,
                 title="SAC — LunarLanderContinuous-v3",
                 experiments_dir=EXPERIMENTS_DIR)

last_ep = results_sac['episode_rewards'][-10:]
print(f"\nAverage reward (last 10 episodes): {np.mean(last_ep):.1f}")
if np.mean(last_ep) >= 200:
    print("LunarLander SOLVED with SAC!")

In [ ]:
# A.10 — Visualize the trained SAC agent
# Record 3 episodes using the deterministic policy (tanh of the mean).
video_path = record_agent_video(
    results_sac['agent'],
    env_name        = "LunarLanderContinuous-v3",
    agent_type      = "sac",
    num_episodes    = 3,
    seed            = 0,
    name_prefix     = "sac_lunar",
    experiments_dir = EXPERIMENTS_DIR,
)
if video_path:
    display(Video(video_path, embed=True, width=500))

In [ ]:
# =====================================================================
# B.1 — A2C vs SAC: Sample Efficiency Comparison
#
# Run each algorithm with 3 seeds on LunarLanderContinuous-v3 and
# plot mean ± std reward to compare sample efficiency.
#
# seeds = [42, 123, 456]
#
# a2c_runs = [train_a2c(env_name="LunarLanderContinuous-v3",
#                        total_timesteps=500_000,
#                        n_steps=2048, batch_size=64, lr=3e-4, seed=s)
#              for s in seeds]
#
# sac_runs = [train_sac(env_name="LunarLanderContinuous-v3",
#                        total_timesteps=300_000,
#                        learning_starts=5000, batch_size=256, lr=3e-4, seed=s)
#              for s in seeds]
#
# plot_comparison(
#     {'A2C': a2c_runs, 'SAC': sac_runs},
#     title="A2C vs SAC — LunarLanderContinuous-v3 (3 seeds)",
#     experiments_dir=EXPERIMENTS_DIR,
# )
#
# For each algorithm print mean ± std of avg reward (last 20 episodes):
#   for label, runs in [('A2C', a2c_runs), ('SAC', sac_runs)]:
#       last20 = [np.mean(r['episode_rewards'][-20:]) for r in runs]
#       print(f"{label}: {np.mean(last20):.1f} ± {np.std(last20):.1f}")
# =====================================================================
raise NotImplementedError("Run A2C vs SAC comparison — B.1")

In [ ]:
# =====================================================================
# B.2 — Ablation: Entropy Temperature in SAC
#
# Compare three fixed α values vs automatic tuning to understand
# how the entropy temperature affects exploration/exploitation.
# Hypothesis: auto-tuning should be at least as good as the best fixed α.
#
# seeds = [42, 123, 456]
# alpha_results = {
#     'Auto α':   [train_sac(env_name="LunarLanderContinuous-v3",
#                             total_timesteps=200_000,
#                             learning_starts=5000, batch_size=256,
#                             autotune_alpha=True, seed=s) for s in seeds],
#     'α = 0.01': [train_sac(..., autotune_alpha=False, alpha=0.01, seed=s) for s in seeds],
#     'α = 0.2':  [train_sac(..., autotune_alpha=False, alpha=0.2,  seed=s) for s in seeds],
#     'α = 1.0':  [train_sac(..., autotune_alpha=False, alpha=1.0,  seed=s) for s in seeds],
# }
# plot_comparison(alpha_results,
#                 title="SAC — Entropy Temperature Ablation",
#                 experiments_dir=EXPERIMENTS_DIR)
#
# Print mean ± std of avg reward (last 20 episodes) for each setting.
# =====================================================================
raise NotImplementedError("Run SAC entropy temperature ablation — B.2")

In [ ]:
# =====================================================================
# B.3 — Ablation: Entropy Temperature in SAC
# Compare autotune vs fixed α values.
#
# seeds = [42, 123, 456]
# alpha_results = {
#     'Auto α':      [train_sac(..., autotune_alpha=True, seed=s)  for s in seeds],
#     'α = 0.01':    [train_sac(..., autotune_alpha=False, alpha=0.01, seed=s) for s in seeds],
#     'α = 0.2':     [train_sac(..., autotune_alpha=False, alpha=0.2, seed=s)  for s in seeds],
#     'α = 1.0':     [train_sac(..., autotune_alpha=False, alpha=1.0, seed=s)  for s in seeds],
# }
# plot_comparison(alpha_results, title="SAC — Entropy Temperature Ablation",
#                 use_timesteps=True)
# =====================================================================
raise NotImplementedError("Run entropy temperature ablation — B.3")

In [ ]:
# =====================================================================
# B.4 — Ablation: GAE Lambda in A2C
# Study the bias-variance trade-off in advantage estimation.
#
# seeds = [42, 123, 456]
# lambda_results = {
#     'λ = 0.0  (1-step TD)': [train_a2c(..., gae_lambda=0.0, seed=s) for s in seeds],
#     'λ = 0.95 (default)':   [train_a2c(..., gae_lambda=0.95, seed=s) for s in seeds],
#     'λ = 1.0  (Monte Carlo)':[train_a2c(..., gae_lambda=1.0, seed=s) for s in seeds],
# }
# plot_comparison(lambda_results, title="A2C — GAE Lambda Ablation")
#
# For each setting, print: mean ± std of avg reward (last 20 episodes)
# =====================================================================
raise NotImplementedError("Run GAE lambda ablation — B.4")

---

## Summary

**TODO:** Write a brief summary of your findings (double-click to edit).

1. What is the key difference between A2C (on-policy) and SAC (off-policy)? How does this affect sample efficiency?
2. How does the entropy temperature α affect SAC's behavior? What happens with α → 0 (deterministic) vs α → ∞ (very stochastic)?
3. What was the effect of GAE lambda on A2C? Which value worked best, and why?
4. Did SAC solve BipedalWalker? How many timesteps, and what challenges did you observe?
5. Which algorithm would you recommend for a new continuous control task, and why?

---

**Lab designed by Amath Sow:** amath.sow@liu.se

**TDDE78 — Deep Reinforcement Learning, Linköping University, Spring 2026**